# Notebook 10 — Age Bias Analysis

Two-part audit of the car price prediction pipeline:

**Part A — Cleaning Audit**: Does brand-level IQR outlier removal disproportionately eliminate old or very new cars?

**Part B — Model Error by Year**: Does the `lean_full` quantile model have higher prediction error for oldest or newest cars in the test set?

Primary metric: `pct_err_over_year_median = mean(|actual − pred|) / median(actual prices in that year) × 100`

In [ ]:
import os
import sys
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# ── Project root & source path ────────────────────────────────────────────────
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import data_processing
from config import DATA_PATH
from features.feature_engineering import CarPriceFeatureEngineer
from data_processing import CarDataProcessor

# ── Key constants (must match notebook 09 exactly) ────────────────────────────
CURRENT_YEAR      = 2025
TRAIN_TEST_SEED   = 42
IQR_MULTIPLIER    = 1.5
ARTIFACTS_DIR     = project_root / "output" / "mlflow_artifacts"
data_dir          = Path(os.path.join(DATA_PATH, "le_boncoin_13_oct_2025"))

# ── Age buckets for grouping car cohorts ──────────────────────────────────────
AGE_BUCKETS = [
    (0,  2,  "0-2 yr\n(2023-25)"),
    (3,  5,  "3-5 yr\n(2020-22)"),
    (6,  10, "6-10 yr\n(2015-19)"),
    (11, 15, "11-15 yr\n(2010-14)"),
    (16, 20, "16-20 yr\n(2005-09)"),
    (21, 99, "21+ yr\n(pre-2005)"),
]
AGE_BUCKET_LABELS = [lab for _, _, lab in AGE_BUCKETS]

def age_to_bucket_label(age: int) -> str:
    for lo, hi, label in AGE_BUCKETS:
        if lo <= age <= hi:
            return label
    return "unknown"

print(f"Project root : {project_root}")
print(f"Artifacts dir: {ARTIFACTS_DIR}")
print(f"Data dir     : {data_dir}")

## Part A — Cleaning Audit

### A2. Load Raw Data & Run Pipeline Up to (but not including) IQR Step

In [ ]:
# Load raw data
df_raw = data_processing.load_car_data(data_dir)
print(f"Raw rows: {len(df_raw):,}")

# Run all cleaning steps EXCEPT IQR outlier removal
# This replicates what _remove_outliers_iqr would receive as input
processor = CarDataProcessor(verbose=False)
df_pre_iqr = processor._convert_data_types(df_raw)
df_pre_iqr = processor._remove_antique_cars(df_pre_iqr)
df_pre_iqr = processor._remove_autre_entries(df_pre_iqr)
df_pre_iqr = processor._clean_horsepower(df_pre_iqr)
df_pre_iqr = processor._filter_rare_brands(df_pre_iqr)

print(f"Pre-IQR rows: {len(df_pre_iqr):,}")
print(f"Columns: {df_pre_iqr.columns.to_list()}")

### A3. Flag IQR Outliers (Replicate `_remove_outliers_iqr` with a flag instead of a filter)

Uses `(price + 1).log()` (log1p) to match the internal IQR logic exactly. Adds `is_iqr_outlier` and `year_bucket` columns without removing any rows.

In [ ]:
# Add log1p price (to match _remove_outliers_iqr exactly) and car_age,
# then filter to the same rows that IQR would consider (valid price, km, year).
df_flagging = (
    df_pre_iqr
    .with_columns([
        (pl.col("price") + 1).log().alias("log_price_iqr"),   # log1p — matches IQR internals
        (CURRENT_YEAR - pl.col("year")).alias("car_age"),
    ])
    .filter(
        pl.col("price").is_not_null()
        & pl.col("km").is_not_null()
        & pl.col("year").is_not_null()
        & (pl.col("price") > 0)
        & (pl.col("km") >= 0)
    )
)

# Compute per-brand IQR bounds (same as _remove_outliers_iqr)
bounds = (
    df_flagging
    .group_by("brand")
    .agg([
        pl.col("log_price_iqr").quantile(0.25).alias("q1_lp"),
        pl.col("log_price_iqr").quantile(0.75).alias("q3_lp"),
        pl.col("km").quantile(0.25).alias("q1_km"),
        pl.col("km").quantile(0.75).alias("q3_km"),
    ])
    .with_columns([
        (pl.col("q3_lp") - pl.col("q1_lp")).alias("iqr_lp"),
        (pl.col("q3_km") - pl.col("q1_km")).alias("iqr_km"),
    ])
    .with_columns([
        (pl.col("q1_lp") - IQR_MULTIPLIER * pl.col("iqr_lp")).alias("lo_lp"),
        (pl.col("q3_lp") + IQR_MULTIPLIER * pl.col("iqr_lp")).alias("hi_lp"),
        (pl.col("q1_km") - IQR_MULTIPLIER * pl.col("iqr_km")).alias("lo_km"),
        (pl.col("q3_km") + IQR_MULTIPLIER * pl.col("iqr_km")).alias("hi_km"),
    ])
)

# Join bounds back and flag outliers
df_flagged = (
    df_flagging
    .join(bounds, on="brand", how="left")
    .with_columns([
        (
            (pl.col("log_price_iqr") < pl.col("lo_lp"))
            | (pl.col("log_price_iqr") > pl.col("hi_lp"))
            | (pl.col("km") < pl.col("lo_km"))
            | (pl.col("km") > pl.col("hi_km"))
        ).alias("is_iqr_outlier"),
        pl.col("car_age")
            .map_elements(age_to_bucket_label, return_dtype=pl.Utf8)
            .alias("year_bucket"),
    ])
)

n_total   = len(df_flagged)
n_dropped = df_flagged["is_iqr_outlier"].sum()
print(f"Pre-IQR rows  : {n_total:,}")
print(f"Flagged as IQR outliers: {n_dropped:,}  ({100*n_dropped/n_total:.1f}%)")
print(f"Kept after IQR         : {n_total - n_dropped:,}")

### A4. Audit Summary Tables

Drop rate by calendar year and by age bucket.

In [ ]:
# ── By calendar year ─────────────────────────────────────────────────────────
year_summary_pl = (
    df_flagged
    .group_by("year")
    .agg([
        pl.len().alias("n_total"),
        pl.col("is_iqr_outlier").sum().alias("n_dropped"),
    ])
    .with_columns(
        (100 * pl.col("n_dropped") / pl.col("n_total")).alias("drop_pct")
    )
    .sort("year")
)
year_summary = year_summary_pl.to_pandas()
print("Drop rate by calendar year:")
print(year_summary.to_string(index=False))

# ── By age bucket ─────────────────────────────────────────────────────────────
bucket_summary = (
    df_flagged
    .group_by("year_bucket")
    .agg([
        pl.len().alias("n_total"),
        pl.col("is_iqr_outlier").sum().alias("n_dropped"),
        pl.col("price").median().alias("median_price_eur"),
    ])
    .with_columns(
        (100 * pl.col("n_dropped") / pl.col("n_total")).alias("drop_pct")
    )
    .to_pandas()
)
# Re-order by AGE_BUCKETS definition
bucket_summary["_order"] = bucket_summary["year_bucket"].map(
    {lab: i for i, (_, _, lab) in enumerate(AGE_BUCKETS)}
)
bucket_summary = bucket_summary.sort_values("_order").drop(columns="_order")
print("\nDrop rate by age bucket:")
print(bucket_summary.to_string(index=False))

### A5. Audit Visualisations

**Chart 1** — Stacked bar of kept vs dropped rows per year, with drop% line overlay  
**Chart 2** — Heatmap: brand × age-bucket IQR drop rate (top 15 brands by volume)  
**Chart 3** — KDE: price distribution of dropped vs kept cars, faceted by age bucket  
**Chart 4** — Boxplot: price (EUR) of dropped vs kept cars per age bucket

In [ ]:
# ── Chart 1: Stacked bar per year + drop% line ───────────────────────────────
fig, ax1 = plt.subplots(figsize=(18, 5))

kept    = year_summary["n_total"] - year_summary["n_dropped"]
dropped = year_summary["n_dropped"]
years   = year_summary["year"]
x       = range(len(years))

ax1.bar(x, kept,    label="Kept",    color="#4C72B0", width=0.7)
ax1.bar(x, dropped, label="Dropped", color="#DD8452", width=0.7, bottom=kept)
ax1.set_xticks(list(x))
ax1.set_xticklabels(years, rotation=45, ha="right", fontsize=9)
ax1.set_ylabel("Number of listings")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(list(x), year_summary["drop_pct"], color="crimson", marker="o",
         linewidth=2, markersize=5, label="Drop %")
ax2.set_ylabel("IQR drop rate (%)", color="crimson")
ax2.tick_params(axis="y", labelcolor="crimson")
ax2.set_ylim(0, max(year_summary["drop_pct"]) * 1.3)
ax2.legend(loc="upper right")

ax1.set_title("IQR Outlier Removal by Calendar Year — Count and Drop Rate", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 2: Brand × age-bucket IQR drop rate heatmap ────────────────────────
# Build pivot: brand (rows) × year_bucket (cols), values = drop_pct
top_brands = (
    df_flagged
    .group_by("brand")
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
    .head(15)["brand"]
    .to_list()
)

heatmap_data = (
    df_flagged
    .filter(pl.col("brand").is_in(top_brands))
    .group_by(["brand", "year_bucket"])
    .agg([
        pl.len().alias("n_total"),
        pl.col("is_iqr_outlier").sum().alias("n_dropped"),
    ])
    .with_columns(
        (100 * pl.col("n_dropped") / pl.col("n_total")).alias("drop_pct")
    )
    .to_pandas()
    .pivot(index="brand", columns="year_bucket", values="drop_pct")
    .reindex(columns=AGE_BUCKET_LABELS)
    .reindex(top_brands)
)

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    heatmap_data, annot=True, fmt=".0f", cmap="YlOrRd",
    linewidths=0.4, ax=ax, vmin=0, vmax=40,
    cbar_kws={"label": "IQR drop rate (%)"},
)
ax.set_title("IQR Drop Rate (%) — Top 15 Brands × Age Bucket", fontsize=13, fontweight="bold")
ax.set_xlabel("Age Bucket")
ax.set_ylabel("Brand")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 3: KDE of log1p price — dropped vs kept, faceted by age bucket ─────
df_plot = df_flagged.select(["log_price_iqr", "is_iqr_outlier", "year_bucket"]).to_pandas()
df_plot["status"] = df_plot["is_iqr_outlier"].map({True: "Dropped", False: "Kept"})

n_buckets = len(AGE_BUCKET_LABELS)
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=False)
axes = axes.flatten()

for i, bucket in enumerate(AGE_BUCKET_LABELS):
    subset = df_plot[df_plot["year_bucket"] == bucket]
    if len(subset) < 5:
        axes[i].set_visible(False)
        continue
    for label, grp in subset.groupby("status"):
        if len(grp) > 5:
            grp["log_price_iqr"].plot.kde(ax=axes[i], label=label,
                                           linewidth=2,
                                           color="#DD8452" if label == "Dropped" else "#4C72B0")
    axes[i].set_title(bucket.replace("\n", " "), fontsize=10)
    axes[i].set_xlabel("log(price + 1)")
    axes[i].legend(fontsize=8)
    axes[i].set_xlim(5, 13)

fig.suptitle("KDE of log(price+1): Dropped vs Kept by Age Bucket", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 4: Boxplot — price EUR dropped vs kept per age bucket ───────────────
df_box = df_flagged.select(["price", "is_iqr_outlier", "year_bucket"]).to_pandas()
df_box["status"] = df_box["is_iqr_outlier"].map({True: "Dropped", False: "Kept"})
df_box["_order"] = df_box["year_bucket"].map(
    {lab: i for i, (_, _, lab) in enumerate(AGE_BUCKETS)}
)
df_box = df_box.sort_values("_order")

fig, ax = plt.subplots(figsize=(16, 6))
palette = {"Kept": "#4C72B0", "Dropped": "#DD8452"}
sns.boxplot(
    data=df_box,
    x="year_bucket", y="price", hue="status",
    order=AGE_BUCKET_LABELS,
    palette=palette,
    showfliers=False,
    ax=ax,
)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"€{v:,.0f}"))
ax.set_title("Price (EUR) Distribution: Dropped vs Kept Cars by Age Bucket\n(log scale, whiskers = 5th–95th pct, no fliers)", fontsize=12, fontweight="bold")
ax.set_xlabel("Age Bucket")
ax.set_ylabel("Price (EUR, log scale)")
ax.legend(title="IQR status")
plt.tight_layout()
plt.show()

---

## Part B — Model Error by Year / Age Bracket

### B1. Reconstruct the Exact Test Set from Notebook 09

Must reproduce the same 80/20 split with `TRAIN_TEST_SEED = 42` and add `log_price` using plain `log()` (not log1p) to match nb09 Cell 3.

In [ ]:
# Run full clean pipeline (includes IQR), then add log_price with plain log()
# to match nb09 Cell 3 exactly.
df = data_processing.clean_car_data(df_raw, verbose=False)
df = df.with_columns(pl.col("price").log().alias("log_price"))

# Reproduce the same train/test split as nb09
train_indices, test_indices = train_test_split(
    range(len(df)), test_size=0.2, random_state=TRAIN_TEST_SEED
)
df_train = df[train_indices]
df_test  = df[test_indices]

# Metadata arrays for grouping
years_test    = df_test["year"].to_numpy()
car_age_test  = CURRENT_YEAR - years_test
y_test_eur    = df_test["price"].to_numpy()
brands_test   = df_test["brand"].to_numpy()

print(f"Clean after IQR : {len(df):,} rows")
print(f"Train: {len(df_train):,}  |  Test: {len(df_test):,}")
print(f"Test year range : {years_test.min()} – {years_test.max()}")
print(f"Test price range: €{y_test_eur.min():,.0f} – €{y_test_eur.max():,.0f}")

### B2. Feature Engineering for `lean_full` Spec

Fits `CarPriceFeatureEngineer` on the training set (no brand/model OHE, no km, no HP, no energie), then transforms both sets. This is the identical configuration used in nb09 for `lean_full`.

In [ ]:
# Identical to nb09 _NO_OHE config for lean_full
fe_full = CarPriceFeatureEngineer(
    current_year=CURRENT_YEAR,
    brand_onehot=False,
    model_onehot=False,
    add_horsepower_features=False,
    add_energie_ohe=False,
)
fe_full.fit(df_train.drop(["price", "log_price"]), df_train["price"])
df_test_fe = fe_full.transform(df_test)

print(f"Feature-engineered test set: {df_test_fe.shape}")
print(f"Sample columns: {df_test_fe.columns[:10]}")

### B3. Load Saved Models and Generate Predictions

Loads q15 / q50 / q85 pkl files from `output/mlflow_artifacts/`. Feature list is read from the model's `feature_name_` attribute (identical to how nb09 does it).

In [ ]:
SPEC_TO_ANALYZE = "lean_full"   # ← change this single variable to swap specs

models = {}
for q_label in ["q15", "q50", "q85"]:
    pkl_path = ARTIFACTS_DIR / f"{SPEC_TO_ANALYZE}_{q_label}_model.pkl"
    models[q_label] = joblib.load(pkl_path)
    print(f"Loaded {pkl_path.name}  ({len(models[q_label].feature_name_)} features)")

# Build X_test using feature names from the q50 model
feat_cols = models["q50"].feature_name_
available  = [c for c in feat_cols if c in df_test_fe.columns]
missing    = [c for c in feat_cols if c not in df_test_fe.columns]
if missing:
    print(f"WARNING: {len(missing)} features missing from df_test_fe: {missing[:5]}")

X_test = df_test_fe.select(available).to_pandas().fillna(0)

# Predict in log-price space, then exponentiate to EUR
y_pred_log = {q: models[q].predict(X_test) for q in models}
y_pred_eur = {q: np.exp(y_pred_log[q]) for q in models}

print(f"\nX_test shape : {X_test.shape}")
print(f"q50 MAE      : €{mean_absolute_error(y_test_eur, y_pred_eur['q50']):,.0f}")

### B4. Metric Helper Functions

Primary metric: `pct_err_over_year_median = mean(|actual − pred|) / median(actual prices in year) × 100`  
Signed bias: `median((pred − actual) / actual × 100)`  
Coverage: fraction of actuals inside the [q15, q85] interval.

In [ ]:
def compute_year_metrics(
    y_true_eur: np.ndarray,
    y_pred_q50_eur: np.ndarray,
    y_pred_q15_eur: np.ndarray,
    y_pred_q85_eur: np.ndarray,
    group_vals: np.ndarray,
    group_label: str = "year",
) -> pd.DataFrame:
    """
    Compute per-group error metrics normalised to that group's median price.

    Returns a DataFrame with columns:
      {group_label}, n, median_price_eur,
      pct_err_over_median  – primary metric
      mae_eur              – raw MAE (informational only)
      mape_pct             – MAPE %
      median_signed_pct    – median signed % error (bias indicator)
      coverage_70          – fraction inside [q15, q85] interval
    """
    rows = []
    for gval in sorted(np.unique(group_vals)):
        mask = group_vals == gval
        if mask.sum() < 3:
            continue
        act  = y_true_eur[mask]
        pred = y_pred_q50_eur[mask]
        lo   = y_pred_q15_eur[mask]
        hi   = y_pred_q85_eur[mask]

        med_price = np.median(act)
        mae_val   = mean_absolute_error(act, pred)
        rows.append({
            group_label:             gval,
            "n":                     int(mask.sum()),
            "median_price_eur":      round(med_price, 0),
            "pct_err_over_median":   round(100 * mae_val / med_price, 2),
            "mae_eur":               round(mae_val, 0),
            "mape_pct":              round(100 * np.mean(np.abs((act - pred) / np.clip(act, 1, None))), 2),
            "median_signed_pct":     round(float(np.median((pred - act) / np.clip(act, 1, None) * 100)), 2),
            "coverage_70":           round(float(np.mean((lo <= act) & (act <= hi))), 4),
        })
    return pd.DataFrame(rows)


def compute_age_bucket_metrics(
    y_true_eur: np.ndarray,
    y_pred_q50_eur: np.ndarray,
    y_pred_q15_eur: np.ndarray,
    y_pred_q85_eur: np.ndarray,
    car_ages: np.ndarray,
) -> pd.DataFrame:
    """Same as compute_year_metrics but groups by AGE_BUCKETS."""
    bucket_labels = np.array([age_to_bucket_label(a) for a in car_ages])
    df = compute_year_metrics(
        y_true_eur, y_pred_q50_eur, y_pred_q15_eur, y_pred_q85_eur,
        bucket_labels, group_label="year_bucket",
    )
    df["_order"] = df["year_bucket"].map(
        {lab: i for i, (_, _, lab) in enumerate(AGE_BUCKETS)}
    )
    return df.sort_values("_order").drop(columns="_order").reset_index(drop=True)


# ── Compute metrics ───────────────────────────────────────────────────────────
year_metrics   = compute_year_metrics(
    y_test_eur, y_pred_eur["q50"], y_pred_eur["q15"], y_pred_eur["q85"],
    years_test, "year",
)
bucket_metrics = compute_age_bucket_metrics(
    y_test_eur, y_pred_eur["q50"], y_pred_eur["q15"], y_pred_eur["q85"],
    car_age_test,
)

print("=== By year (first 10 rows) ===")
print(year_metrics.head(10).to_string(index=False))
print("\n=== By age bucket ===")
print(bucket_metrics.to_string(index=False))

### B5. Performance Visualisations

**Chart 5** — `pct_err_over_median` by calendar year (line + confidence band)  
**Chart 6** — Signed bias (median % over/under) by year  
**Chart 7** — Interval coverage (q15–q85) by year  
**Chart 8** — Age-bucket bar chart: `pct_err_over_median` + count overlay  
**Chart 9** — Signed bias by age bucket (horizontal bar)

In [ ]:
# ── Chart 5: pct_err_over_median by year ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(year_metrics["year"], year_metrics["pct_err_over_median"],
        marker="o", linewidth=2.0, color="#4C72B0", markersize=6, label=SPEC_TO_ANALYZE)

# Annotate n for context
for _, row in year_metrics.iterrows():
    ax.annotate(
        f"n={int(row['n'])}",
        (row["year"], row["pct_err_over_median"]),
        textcoords="offset points", xytext=(0, 8),
        fontsize=7, ha="center", color="grey",
    )

ax.axhline(year_metrics["pct_err_over_median"].mean(), color="crimson",
           linestyle="--", linewidth=1.2, label="Overall mean")
ax.set_xlabel("Car Year")
ax.set_ylabel("% Error over Year Median Price")
ax.set_title(f"[{SPEC_TO_ANALYZE}] Normalised Error (pct_err_over_year_median) by Calendar Year",
             fontsize=12, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 6: Signed bias (median % over/under) by year ───────────────────────
fig, ax = plt.subplots(figsize=(16, 4))

colors = ["#DD8452" if v > 0 else "#4C72B0" for v in year_metrics["median_signed_pct"]]
ax.bar(year_metrics["year"], year_metrics["median_signed_pct"], color=colors, width=0.7, alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Car Year")
ax.set_ylabel("Median signed % error\n(+ve = over-predict)")
ax.set_title(f"[{SPEC_TO_ANALYZE}] Prediction Bias by Calendar Year\n"
             "Positive = model over-predicts, Negative = under-predicts",
             fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 7: Interval coverage (q15–q85) by year ─────────────────────────────
# Target coverage for a 70% interval is ~0.70
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(year_metrics["year"], year_metrics["coverage_70"] * 100,
        marker="s", linewidth=2.0, color="#55A868", markersize=6)
ax.axhline(70, color="crimson", linestyle="--", linewidth=1.2, label="Ideal 70% coverage")
ax.set_xlabel("Car Year")
ax.set_ylabel("Coverage (%)")
ax.set_ylim(0, 110)
ax.set_title(f"[{SPEC_TO_ANALYZE}] Q15–Q85 Interval Coverage by Year\n"
             "Should be close to 70% for a well-calibrated interval",
             fontsize=12, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 8: Age-bucket bar — pct_err_over_median + sample count overlay ─────
fig, ax1 = plt.subplots(figsize=(14, 5))

xlabels = [b.replace("\n", " ") for b in bucket_metrics["year_bucket"]]
x       = range(len(xlabels))
bars    = ax1.bar(x, bucket_metrics["pct_err_over_median"], color="#4C72B0", width=0.6, alpha=0.85)
ax1.set_xticks(list(x))
ax1.set_xticklabels(xlabels, fontsize=10)
ax1.set_ylabel("% Error over Age-Bucket Median Price", fontsize=11)
ax1.set_title(f"[{SPEC_TO_ANALYZE}] Normalised Error by Car Age Bracket",
              fontsize=12, fontweight="bold")

# Annotate bar values
for bar, val in zip(bars, bucket_metrics["pct_err_over_median"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
             f"{val:.1f}%", ha="center", va="bottom", fontsize=9.5, fontweight="bold")

ax2 = ax1.twinx()
ax2.plot(list(x), bucket_metrics["n"], color="darkorange",
         marker="D", linewidth=2, markersize=7, label="n (test samples)")
ax2.set_ylabel("Test set sample count", color="darkorange")
ax2.tick_params(axis="y", labelcolor="darkorange")
ax2.legend(loc="upper right")

# Overall mean reference
overall_mean = (bucket_metrics["pct_err_over_median"] * bucket_metrics["n"]).sum() / bucket_metrics["n"].sum()
ax1.axhline(overall_mean, color="crimson", linestyle="--", linewidth=1.2,
            label=f"Weighted mean {overall_mean:.1f}%")
ax1.legend(loc="upper left")
ax1.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 9: Signed bias by age bucket (horizontal diverging bar) ─────────────
fig, ax = plt.subplots(figsize=(10, 5))

colors = ["#DD8452" if v > 0 else "#4C72B0" for v in bucket_metrics["median_signed_pct"]]
xlabels_h = [b.replace("\n", " ") for b in bucket_metrics["year_bucket"]]
y_pos = range(len(xlabels_h))

ax.barh(list(y_pos), bucket_metrics["median_signed_pct"], color=colors, alpha=0.85, height=0.55)
ax.axvline(0, color="black", linewidth=0.9)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(xlabels_h, fontsize=10)
ax.set_xlabel("Median signed % error  (+ve = model over-predicts)", fontsize=10)
ax.set_title(f"[{SPEC_TO_ANALYZE}] Prediction Bias by Age Bracket\n"
             "Orange = over-predict, Blue = under-predict",
             fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 10: Scatter — actual vs predicted, coloured by age bucket ───────────
bucket_labels_test = np.array([age_to_bucket_label(a) for a in car_age_test])
palette_buckets    = sns.color_palette("tab10", n_colors=len(AGE_BUCKET_LABELS))
color_map          = {lab: col for lab, col in zip(AGE_BUCKET_LABELS, palette_buckets)}

fig, ax = plt.subplots(figsize=(10, 8))

for bucket in AGE_BUCKET_LABELS:
    mask = bucket_labels_test == bucket
    if mask.sum() == 0:
        continue
    ax.scatter(
        y_test_eur[mask], y_pred_eur["q50"][mask],
        alpha=0.3, s=8,
        color=color_map[bucket],
        label=bucket.replace("\n", " "),
    )

lims = [min(y_test_eur.min(), y_pred_eur["q50"].min()),
        max(y_test_eur.max(), y_pred_eur["q50"].max())]
ax.plot(lims, lims, 'k--', linewidth=1.0, alpha=0.6, label="Perfect prediction")
ax.set_xscale("log")
ax.set_yscale("log")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"€{v:,.0f}"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"€{v:,.0f}"))
ax.set_xlabel("Actual price (EUR, log scale)")
ax.set_ylabel("Predicted price q50 (EUR, log scale)")
ax.set_title(f"[{SPEC_TO_ANALYZE}] Actual vs Predicted — coloured by Age Bucket",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8, markerscale=3, loc="upper left")
plt.tight_layout()
plt.show()

---

## Part C — Multi-Spec Comparison (Optional Extension)

Run the same age-bucket error analysis for multiple specs to see whether adding km/HP/energie features improves performance in particular age cohorts.

To activate, add more spec names to `SPECS_TO_COMPARE`. Only specs with matching `_NO_OHE` feature engineering work out-of-the-box; OHE specs (lean_ohe, lean_plus_*) require rebuilding their specific FE transforms and are tagged below.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Add spec names to this list to compare multiple models side-by-side.
# Specs that share the same _NO_OHE feature transform (df_test_fe from B2):
#   lean_full, extended_km, extended_km_energie_hp
# OHE specs need a dedicated FE pass — not wired up here yet.
# ─────────────────────────────────────────────────────────────────────────────
SPECS_TO_COMPARE = [
    "lean_full",
    # "extended_km",
    # "extended_km_energie_hp",
]

# Specs that use the same fe_full transform (no OHE, no km columns in feature list)
# Specs that need km columns: we need a fe_km transform fitted with add_km=True
# For simplicity we use the same df_test_fe for all _NO_OHE variants;
# LightGBM will simply zero-fill missing columns via fillna(0) in X_test.
multi_results: dict[str, pd.DataFrame] = {}

for spec in SPECS_TO_COMPARE:
    try:
        ms = {}
        for q_label in ["q15", "q50", "q85"]:
            pkl_path = ARTIFACTS_DIR / f"{spec}_{q_label}_model.pkl"
            ms[q_label] = joblib.load(pkl_path)

        feat_cols_s = ms["q50"].feature_name_
        X_te_s = df_test_fe.select(
            [c for c in feat_cols_s if c in df_test_fe.columns]
        ).to_pandas().fillna(0)

        preds = {q: np.exp(ms[q].predict(X_te_s)) for q in ms}

        bucket_m = compute_age_bucket_metrics(
            y_test_eur, preds["q50"], preds["q15"], preds["q85"], car_age_test
        )
        bucket_m["spec"] = spec
        multi_results[spec] = bucket_m
        print(f"✓  {spec:<35}  overall pct_err = "
              f"{(bucket_m['pct_err_over_median'] * bucket_m['n']).sum() / bucket_m['n'].sum():.2f}%")
    except FileNotFoundError:
        print(f"✗  {spec}: model pkl not found, skipping")

print(f"\nSpecs loaded: {list(multi_results.keys())}")

In [ ]:
# ── Chart 11: Multi-spec grouped bar — pct_err_over_median by age bucket ──────
if len(multi_results) > 0:
    df_multi = pd.concat(multi_results.values(), ignore_index=True)
    pivot = df_multi.pivot(index="year_bucket", columns="spec", values="pct_err_over_median")
    pivot = pivot.reindex(AGE_BUCKET_LABELS)

    x     = np.arange(len(AGE_BUCKET_LABELS))
    width = 0.8 / max(len(multi_results), 1)
    specs = list(multi_results.keys())
    colors_ms = sns.color_palette("tab10", n_colors=len(specs))

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, spec in enumerate(specs):
        ax.bar(x + i * width - (len(specs) - 1) * width / 2,
               pivot[spec],
               width=width * 0.9,
               label=spec,
               color=colors_ms[i],
               alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([b.replace("\n", " ") for b in AGE_BUCKET_LABELS], fontsize=9)
    ax.set_ylabel("% Error over Age-Bucket Median Price")
    ax.set_title("Multi-Spec Comparison — Normalised Error by Age Bracket",
                 fontsize=12, fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.4)
    plt.tight_layout()
    plt.show()
else:
    print("No results to plot — add specs to SPECS_TO_COMPARE.")

---

## Summary

| Section | What was checked | Key output |
|---|---|---|
| **A — Cleaning audit** | IQR drop rate vs year / brand / age bracket | Charts 1–4; summary tables |
| **B — Model error by year** | `pct_err_over_year_median`, bias, coverage per year and bracket | Charts 5–10 |
| **C — Multi-spec comparison** | Same metrics across `SPECS_TO_COMPARE` | Chart 11 |

**To add more specs to Part C**, uncomment lines in `SPECS_TO_COMPARE`. Specs that need km features (`extended_km`, `extended_km_energie_hp`) will use zero-filled km columns from the lean FE transform — results will underperform their true potential until a dedicated `fe_km` transform is fitted.